# Лабораторная работа №14: Управляемая генерация изображений

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.

## Цели обучения

После работы студент должен уметь:

- Получить canny/depth/pose controls.
- Сравнить control scale и CFG.
- Использовать control как synthetic annotation для downstream model.

## Теоретическая часть

ControlNet добавляет trainable branch к замороженной diffusion model и принимает spatial condition (edges, depth, pose). В лабораторной используется готовый ControlNet; изменяются conditioning scale и CFG.

![Пример результата ControlNet с Canny-контролем](https://huggingface.co/lllyasviel/sd-controlnet-canny/resolve/main/images/bird_canny_out.png)


Готовый ControlNet добавляет пространственное условие к SD1.5. В работе контур Canny вычисляется только из train MNIST; тот же исходный label сопровождает synthetic image. Edge F1, CLIP text similarity и downstream accuracy на реальном test отражают разные свойства и могут конфликтовать.

Для контуров $F_1=2PR/(P+R)$; допустимое геометрическое отклонение задаётся расширением контуров до расчёта точности и полноты. Downstream accuracy измеряется только на реальном отложенном test.

### Экспериментальный протокол

Выбрано условие Canny, обучение ControlNet не требуется. Полный режим: 20 train контуров каждого из 10 классов, 5 генераций на контур (base + сетка scale/CFG), отдельный real test и одинаковые 500 optimizer steps каждого downstream classifier. Это дорогой эксперимент: перед запуском оцените лимит Colab, сохраняйте artifacts в Drive; FAST_DEV_RUN генерирует по 1 примеру каждого класса и 2 шага классификатора, только smoke. Нужны CUDA, diffusers transformers accelerate opencv-python torchvision.

### Материалы

- [Diffusers ControlNet](https://huggingface.co/docs/diffusers/using-diffusers/controlnet)
- [ControlNet paper](https://arxiv.org/abs/2302.05543)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.

Версии библиотек, источник данных и лицензия должны быть указаны в отчёте.

## Реализация

`FAST_DEV_RUN=True` проверяет связность пайплайна на малом бюджете и не является итоговым экспериментом. Полная сдача требует `FAST_DEV_RUN=False` и реально полученных метрик.

In [1]:
from pathlib import Path
import os, json, random, time
os.environ.setdefault('HF_HUB_DISABLE_XET', '1')  # xet-загрузчик не нужен, классический CDN надёжнее за прокси
import numpy as np
import torch
SEED = 42
FAST_DEV_RUN = False  # True только для короткой проверки связности, не для сдачи.
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True   # одинаковые числа при повторном прогоне
torch.backends.cudnn.benchmark = False
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
ARTIFACTS = Path(os.environ.get('CV_ARTIFACTS', 'artifacts'))
ARTIFACTS.mkdir(parents=True, exist_ok=True)
DATA_ROOT = Path(os.environ.get('CV_COURSE_DATA', str(Path.cwd().parents[1] / 'data')))
DATA_ROOT.mkdir(parents=True, exist_ok=True)
def save_json(name, value):
    destination = ARTIFACTS / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")  # UTF-8 обязательно (Windows-локаль cp1251)
print({'device':DEVICE,'fast_dev_run':FAST_DEV_RUN,'artifacts':str(ARTIFACTS)})

{'device': 'cuda', 'fast_dev_run': False, 'artifacts': 'artifacts'}


In [2]:
# Сеть: если HTTPS перехватывается прокси/антивирусом (self-signed сертификат),
# переводим проверку сертификатов на системное хранилище ОС (пакет truststore,
# однократно: pip install truststore). Если пакета нет — стандартная проверка.
try:
    import truststore, urllib3
    truststore.inject_into_ssl()
    urllib3.disable_warnings()
    print("SSL: проверка сертификатов через системное хранилище ОС (truststore)")
except ImportError:
    print("SSL: truststore не установлен — стандартная проверка сертификатов")

SSL: проверка сертификатов через системное хранилище ОС (truststore)


In [3]:
# Проверка зависимостей лабы 14: diffusers, transformers, accelerate, opencv (cv2), torchvision.
# При отсутствии: pip install -q "diffusers>=0.27" transformers accelerate opencv-python torchvision
# (при проблемах с сертификатами добавьте --trusted-host pypi.org --trusted-host files.pythonhosted.org)
import importlib.util
_need = {"diffusers": '"diffusers>=0.27"', "transformers": '"transformers>=4.40"', "accelerate": "accelerate",
         "cv2": "opencv-python", "torchvision": "torchvision"}
missing = [k for k in _need if importlib.util.find_spec(k) is None]
if missing:
    raise ImportError("Установите недостающее и перезапустите ячейку:  pip install " + " ".join(_need[k] for k in missing))
import torch, diffusers, transformers, torchvision, cv2
_info = {"torch": torch.__version__, "diffusers": diffusers.__version__,
         "transformers": transformers.__version__, "torchvision": torchvision.__version__,
         "opencv": cv2.__version__}
if torch.cuda.is_available():
    _p = torch.cuda.get_device_properties(0)
    _free, _total = torch.cuda.mem_get_info()
    _info["gpu"] = f"{_p.name} | VRAM {_total/2**30:.1f} ГБ, свободно {_free/2**30:.1f} ГБ | cc {_p.major}.{_p.minor}"
    if _p.major < 7:
        _info["gpu_note"] = "Pascal: bf16 недоступен, используем fp16"
    _info["vram_tip"] = "генерация 256px batch 1 + slicing помещается в 4 ГБ; при OOM закройте браузер и другие ядра Jupyter"
else:
    _info["gpu"] = "CUDA недоступна — генерация и обучение невозможны; установите PyTorch с CUDA"
print("Версии:", _info)

Версии: {'torch': '2.14.1+cu126', 'diffusers': '0.39.0', 'transformers': '4.57.6', 'torchvision': '0.29.1+cu126', 'opencv': '5.0.0', 'gpu': 'NVIDIA GeForce GTX 1050 Ti | VRAM 4.0 ГБ, свободно 3.3 ГБ | cc 6.1', 'gpu_note': 'Pascal: bf16 недоступен, используем fp16', 'vram_tip': 'генерация 256px batch 1 + slicing помещается в 4 ГБ; при OOM закройте браузер и другие ядра Jupyter'}


### Непересекающиеся MNIST выборки

*Вес: 1 балл.*

**Постановка.** Контур и метка должны происходить из одной исходной сцены, иначе synthetic augmentation получает неверную разметку. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Загрузите только train/test MNIST, выберите реальные train индексы по классам.

**Результат.** Проверяемое состояние «Непересекающиеся MNIST выборки» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Загрузите только train/test MNIST, выберите реальные train индексы по классам» и его результат доступен следующему этапу.

In [4]:
# ЗАДАНИЕ 1 — непересекающиеся MNIST выборки: train/test, фиксированный выбор real-индексов по классам.
# Контуры (источник синтетики) и real-подвыборка классификатора — ДВЕ НЕПЕРЕСЕКАЮЩИЕСЯ части train;
# test участвует только в финальном сравнении (ЗАДАНИЕ 10) и ни на что больше не влияет.
from torchvision import datasets

MNIST_ROOT = Path(os.environ.get('CV_MNIST_ROOT', str(DATA_ROOT / 'mnist')))
N_CTRL_PER_CLASS = 1 if FAST_DEV_RUN else 20     # контуров на класс (источник синтетики)
N_REAL_PER_CLASS = 2 if FAST_DEV_RUN else 100    # real-подвыборка классификатора

MNIST_TRAIN = datasets.MNIST(str(MNIST_ROOT), train=True, download=True)
MNIST_TEST = datasets.MNIST(str(MNIST_ROOT), train=False, download=True)
_y_train = np.array(MNIST_TRAIN.targets)
assert len(MNIST_TRAIN) == 60000 and len(MNIST_TEST) == 10000, 'неожиданный размер MNIST'

_rng = np.random.RandomState(SEED)               # фиксированный выбор индексов (детерминизм)
_ctrl_parts, _real_parts = [], []
for _c in range(10):
    _perm = _rng.permutation(np.where(_y_train == _c)[0])
    _ctrl_parts.append(_perm[:N_CTRL_PER_CLASS])
    _real_parts.append(_perm[N_CTRL_PER_CLASS:N_CTRL_PER_CLASS + N_REAL_PER_CLASS])
CTRL_IDX = np.concatenate(_ctrl_parts)           # 200 (или 10 в FAST) индексов-источников контуров
REAL_IDX = np.concatenate(_real_parts)           # 1000 (или 20) real-индексов классификатора
assert len(set(CTRL_IDX.tolist()) & set(REAL_IDX.tolist())) == 0, 'подмножества пересекаются'
assert np.array_equal(_y_train[CTRL_IDX], np.repeat(np.arange(10), N_CTRL_PER_CLASS))
assert np.array_equal(_y_train[REAL_IDX], np.repeat(np.arange(10), N_REAL_PER_CLASS))

split_report = {'seed': SEED, 'n_ctrl_per_class': N_CTRL_PER_CLASS, 'n_real_per_class': N_REAL_PER_CLASS,
                'train_size': len(MNIST_TRAIN), 'test_size': len(MNIST_TEST),
                'ctrl_indices': CTRL_IDX.tolist(), 'real_indices': REAL_IDX.tolist(),
                'disjoint': True,
                'note': 'test MNIST используется только в ЗАДАНИИ 10; контуры и real-подвыборка не пересекаются'}
save_json('lab14_mnist_split.json', split_report)
print(f"ЗАДАНИЕ 1 OK: MNIST train ({len(MNIST_TRAIN)}) / test ({len(MNIST_TEST)}) загружены; контуры "
      f"{len(CTRL_IDX)} ({N_CTRL_PER_CLASS}/класс) и real-подвыборка {len(REAL_IDX)} "
      f"({N_REAL_PER_CLASS}/класс) не пересекаются; seed {SEED} -> artifacts/lab14_mnist_split.json")

ЗАДАНИЕ 1 OK: MNIST train (60000) / test (10000) загружены; контуры 200 (20/класс) и real-подвыборка 1000 (100/класс) не пересекаются; seed 42 -> artifacts/lab14_mnist_split.json


C:\Users\eric1\AppData\Local\Temp\ipykernel_3276\123284728.py:12: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  _y_train = np.array(MNIST_TRAIN.targets)


### Контуры Canny и происхождение меток

*Вес: 1 балл.*

**Постановка.** Контур и метка должны происходить из одной исходной сцены, иначе synthetic augmentation получает неверную разметку. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Постройте PNG-контуры, сохраните исходные индексы, классы и фиксированные seeds.

**Результат.** Проверяемое состояние «Контуры Canny и происхождение меток» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Постройте PNG-контуры, сохраните исходные индексы, классы и фиксированные seeds» и его результат доступен следующему этапу.

In [5]:
# ЗАДАНИЕ 2 — контуры Canny: строятся ТОЛЬКО из train MNIST (28px -> upscale 256, Gaussian, Canny);
# PNG + манифест: исходный индекс, класс и фиксированные seeds (один seed контура на все настройки).
import cv2

CTRL_RES = int(os.environ.get('CV_CN_RES', '256'))   # SD1.5 работает и на 256 — экономия VRAM/времени на 4 ГБ
CANNY_LOW, CANNY_HIGH = 60, 160
GEN_STEPS = int(os.environ.get('CV_SD_STEPS', '4' if FAST_DEV_RUN else '20'))
SETTINGS = [{'name': 'base', 'scale': 0.0, 'cfg': 7.5},   # baseline: ControlNet-ветка отключена (scale=0)
            {'name': 's05_cfg30', 'scale': 0.5, 'cfg': 3.0},
            {'name': 's05_cfg75', 'scale': 0.5, 'cfg': 7.5},
            {'name': 's10_cfg30', 'scale': 1.0, 'cfg': 3.0},
            {'name': 's10_cfg75', 'scale': 1.0, 'cfg': 7.5}]

def canny_from_mnist(arr28):
    big = cv2.resize(arr28, (CTRL_RES, CTRL_RES), interpolation=cv2.INTER_CUBIC)
    big = cv2.GaussianBlur(big, (3, 3), 0)
    return cv2.Canny(big, CANNY_LOW, CANNY_HIGH)     # uint8: белые линии на чёрном (конвенция ControlNet-canny)

CTRL_DIR = ARTIFACTS / 'lab14' / 'ctrl'
CTRL_DIR.mkdir(parents=True, exist_ok=True)
ctrl_manifest, _min_edges, t0 = [], 10 ** 9, time.time()
for _k, _i in enumerate(CTRL_IDX):
    _label = int(_y_train[_i])
    edges = canny_from_mnist(MNIST_TRAIN.data[_i].numpy().astype(np.uint8))
    _n = int((edges > 0).sum())
    assert edges.shape == (CTRL_RES, CTRL_RES) and _n > 50, f'пустой контур для train-индекса {_i}'
    _min_edges = min(_min_edges, _n)
    fname = f'ctrl_{_k:03d}_label{_label}.png'
    cv2.imwrite(str(CTRL_DIR / fname), edges)
    ctrl_manifest.append({'contour_id': _k, 'file_name': fname, 'source_index': int(_i), 'label': _label,
                          'seed': SEED + 10_000_003 + _k * 1_009,   # фиксированный seed контура, общий для всех настроек
                          'resolution': CTRL_RES, 'settings': [s['name'] for s in SETTINGS]})
(ARTIFACTS / 'lab14' / 'ctrl_manifest.jsonl').write_text(
    ''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in ctrl_manifest), encoding='utf-8')
save_json('lab14_ctrl_report.json', {'n_contours': len(ctrl_manifest), 'resolution': CTRL_RES,
                                     'canny_low': CANNY_LOW, 'canny_high': CANNY_HIGH,
                                     'min_edge_pixels': _min_edges,
                                     'label_provenance': 'контур и метка из одного train-изображения (source_index)'})
print(f"ЗАДАНИЕ 2 OK: {len(ctrl_manifest)} контуров Canny {CTRL_RES}px (пороги {CANNY_LOW}/{CANNY_HIGH}, "
      f"минимум {_min_edges} edge-пикселей); PNG -> artifacts/lab14/ctrl/, манифест с исходными индексами, "
      f"классами и seeds -> lab14_ctrl_manifest.jsonl; {round(time.time() - t0, 1)} c")

ЗАДАНИЕ 2 OK: 200 контуров Canny 256px (пороги 60/160, минимум 388 edge-пикселей); PNG -> artifacts/lab14/ctrl/, манифест с исходными индексами, классами и seeds -> lab14_ctrl_manifest.jsonl; 0.2 c


### Загрузка предобученного ControlNet

*Вес: 1 балл.*

**Постановка.** Совместное изменение силы контроля и CFG выявляет компромисс между структурой контура и текстовым условием. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Загрузите SD1.5 и Canny ControlNet с совместимым scheduler; CPU-имитация недопустима.

**Результат.** Проверяемое состояние «Загрузка предобученного ControlNet» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Загрузите SD1.5 и Canny ControlNet с совместимым scheduler; CPU-имитация недопустима» и его результат доступен следующему этапу.

In [6]:
# ЗАДАНИЕ 3 — SD1.5 + ControlNet Canny с совместимым scheduler (UniPC); fp16; CPU-имитация недопустима.
import gc
from diffusers import ControlNetModel, StableDiffusionControlNetPipeline, UniPCMultistepScheduler

SD_ID = os.getenv('CV_SD_MODEL', 'stable-diffusion-v1-5/stable-diffusion-v1-5')
CN_ID = os.getenv('CV_CN_MODEL', 'lllyasviel/sd-controlnet-canny')
assert torch.cuda.is_available(), 'Нужен GPU (см. постановку): CPU-имитация генерации запрещена.'
DEV = 'cuda'

_free_b, _total_b = torch.cuda.mem_get_info()
print(f'VRAM до загрузки: свободно {_free_b/2**30:.1f} ГБ из {_total_b/2**30:.1f} ГБ')
controlnet = ControlNetModel.from_pretrained(CN_ID, torch_dtype=torch.float16)
try:
    SD_CN_PIPE = StableDiffusionControlNetPipeline.from_pretrained(
        SD_ID, controlnet=controlnet, torch_dtype=torch.float16,
        safety_checker=None, requires_safety_checker=False, variant='fp16')
    print('SD1.5 + ControlNet: загружены fp16-веса (variant=fp16)')
except Exception as e:
    print(f'variant=fp16 недоступен ({type(e).__name__}) — загружаем основные веса')
    SD_CN_PIPE = StableDiffusionControlNetPipeline.from_pretrained(
        SD_ID, controlnet=controlnet, torch_dtype=torch.float16,
        safety_checker=None, requires_safety_checker=False)
SD_CN_PIPE.scheduler = UniPCMultistepScheduler.from_config(SD_CN_PIPE.scheduler.config)  # совместимый быстрый scheduler
SD_CN_PIPE.set_progress_bar_config(disable=True)
SD_CN_PIPE.enable_attention_slicing()
SD_CN_PIPE.enable_vae_slicing()

# --- кэш text-эмбеддингов: 10 промптов классов + пустой (unconditional, нужен для CFG) ---
CLASS_WORDS = ['zero', 'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine']
PROMPTS = [f'a photo of a handwritten digit {w}, white chalk strokes on a plain black background, centered'
           for w in CLASS_WORDS]
SD_CN_PIPE.text_encoder.to(DEV)
def _encode(texts):
    _ids = SD_CN_PIPE.tokenizer(list(texts), padding='max_length', truncation=True,
                                max_length=77, return_tensors='pt').input_ids.to(DEV)
    with torch.no_grad():
        return SD_CN_PIPE.text_encoder(_ids)[0].detach().cpu()
_cond = _encode(PROMPTS)
TEXT_EMBEDS = {p: _cond[i] for i, p in enumerate(PROMPTS)}
UNCOND_EMBED = _encode([''])[0]
SD_CN_PIPE.text_encoder.to('cpu')
SD_CN_PIPE.unet.to(DEV); SD_CN_PIPE.vae.to(DEV); SD_CN_PIPE.controlnet.to(DEV)
gc.collect(); torch.cuda.empty_cache()

_free_a, _ = torch.cuda.mem_get_info()
save_json('lab14_controlnet_report.json', {
    'sd_model': SD_ID, 'controlnet': CN_ID, 'scheduler': type(SD_CN_PIPE.scheduler).__name__,
    'dtype': 'fp16', 'resolution': CTRL_RES, 'steps': GEN_STEPS,
    'vram_free_before_gb': round(_free_b / 2**30, 2), 'vram_free_after_gb': round(_free_a / 2**30, 2),
    'prompts_cached': len(TEXT_EMBEDS),
    'licenses': {'mnist': 'public domain (Yann LeCun / NYU; раздаётся зеркалами torchvision)',
                 'sd1.5': 'CreativeML Open RAIL-M', 'sd-controlnet-canny': 'OpenRAIL (+ код Apache-2.0)'}})
print(f"ЗАДАНИЕ 3 OK: SD1.5 + ControlNet на {DEV} (fp16), scheduler {type(SD_CN_PIPE.scheduler).__name__}, "
      f"{CTRL_RES}px, {GEN_STEPS} шагов; text-эмбеддинги закэшированы ({len(TEXT_EMBEDS)}+1, text-encoder на CPU); "
      f"VRAM свободно после: {_free_a/2**30:.1f} ГБ")

W1028 18:30:24.507000 3276 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1028 18:30:24.574000 3276 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`


VRAM до загрузки: свободно 3.3 ГБ из 4.0 ГБ


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

SD1.5 + ControlNet: загружены fp16-веса (variant=fp16)


c:\Users\eric1\.conda\envs\fmi-cv\Lib\site-packages\diffusers\pipelines\pipeline_utils.py:2273: FutureWarning: `enable_vae_slicing` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_slicing()` on a `StableDiffusionControlNetPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_slicing()`.
  deprecate(


ЗАДАНИЕ 3 OK: SD1.5 + ControlNet на cuda (fp16), scheduler UniPCMultistepScheduler, 256px, 20 шагов; text-эмбеддинги закэшированы (10+1, text-encoder на CPU); VRAM свободно после: 0.0 ГБ


### Факторная сетка генераций

*Вес: 1 балл.*

**Постановка.** Совместное изменение силы контроля и CFG выявляет компромисс между структурой контура и текстовым условием. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Сравните baseline и все сочетания силы контроля и CFG на одинаковых входах и seeds.

**Результат.** Проверяемое состояние «Факторная сетка генераций» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Сравните baseline и все сочетания силы контроля и CFG на одинаковых входах и seeds» и его результат доступен следующему этапу.

In [7]:
# ЗАДАНИЕ 4 — факторная сетка: base (scale=0) и 4 сочетания scale x CFG на ОДИНАКОВЫХ входах и seeds.
# У одного контура один seed на все настройки: различия поколений = только эффект scale/CFG.
from PIL import Image

GEN_DIR = ARTIFACTS / 'lab14' / 'gen'
gen_manifest, t0 = [], time.time()
_total = len(ctrl_manifest) * len(SETTINGS)
_done = 0
for _row in ctrl_manifest:
    _cid, _label, _seed = _row['contour_id'], _row['label'], _row['seed']
    _ctrl_img = Image.open(CTRL_DIR / _row['file_name']).convert('RGB')
    _emb = TEXT_EMBEDS[PROMPTS[_label]]
    for _s in SETTINGS:
        _done += 1
        _out = GEN_DIR / _s['name']
        _out.mkdir(parents=True, exist_ok=True)
        _path = _out / f'c{_cid:03d}_seed{_seed}.png'
        _ts = time.time()
        if not _path.exists():      # кэш/resume: перезапуск не перегенерирует существующее
            _g = torch.Generator(device=DEV); _g.manual_seed(_seed)
            with torch.no_grad():
                _img = SD_CN_PIPE(prompt_embeds=_emb[None].to(DEV, torch.float16),
                                  negative_prompt_embeds=UNCOND_EMBED[None].to(DEV, torch.float16),
                                  image=_ctrl_img, num_inference_steps=GEN_STEPS,
                                  guidance_scale=_s['cfg'], controlnet_conditioning_scale=_s['scale'],
                                  height=CTRL_RES, width=CTRL_RES, generator=_g).images[0]
            _img.save(_path)
        gen_manifest.append({'contour_id': _cid, 'label': _label, 'source_index': _row['source_index'],
                             'setting': _s['name'], 'scale': _s['scale'], 'cfg': _s['cfg'], 'seed': _seed,
                             'prompt': PROMPTS[_label], 'path': str(_path),
                             'seconds': round(time.time() - _ts, 1)})
        if _done % 25 == 0 or _done == _total:
            _rate = (time.time() - t0) / _done
            print(f'[сетка] {_done}/{_total} | {_rate:.1f} c/генерация | '
                  f'ETA {(_total - _done) * _rate / 60:.0f} мин')
(ARTIFACTS / 'lab14' / 'gen_manifest.jsonl').write_text(
    ''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in gen_manifest), encoding='utf-8')
print(f"ЗАДАНИЕ 4 OK: {len(gen_manifest)} генераций ({len(ctrl_manifest)} контуров x {len(SETTINGS)} настройки: "
      f"base + scale {{0.5, 1.0}} x CFG {{3.0, 7.5}}), {round(time.time() - t0)} c; одинаковые seeds внутри контура; "
      f"PNG -> artifacts/lab14/gen/<настройка>/, манифест -> lab14_gen_manifest.jsonl")

[сетка] 25/1000 | 0.0 c/генерация | ETA 0 мин
[сетка] 50/1000 | 0.0 c/генерация | ETA 0 мин
[сетка] 75/1000 | 6.2 c/генерация | ETA 96 мин
[сетка] 100/1000 | 9.6 c/генерация | ETA 144 мин
[сетка] 125/1000 | 11.6 c/генерация | ETA 169 мин
[сетка] 150/1000 | 13.0 c/генерация | ETA 184 мин
[сетка] 175/1000 | 13.9 c/генерация | ETA 191 мин
[сетка] 200/1000 | 14.7 c/генерация | ETA 195 мин
[сетка] 225/1000 | 15.3 c/генерация | ETA 198 мин
[сетка] 250/1000 | 16.0 c/генерация | ETA 200 мин
[сетка] 275/1000 | 16.6 c/генерация | ETA 200 мин
[сетка] 300/1000 | 17.0 c/генерация | ETA 199 мин
[сетка] 325/1000 | 17.4 c/генерация | ETA 196 мин
[сетка] 350/1000 | 17.8 c/генерация | ETA 192 мин
[сетка] 375/1000 | 18.1 c/генерация | ETA 188 мин
[сетка] 400/1000 | 18.3 c/генерация | ETA 183 мин
[сетка] 425/1000 | 18.4 c/генерация | ETA 177 мин
[сетка] 450/1000 | 18.7 c/генерация | ETA 171 мин
[сетка] 475/1000 | 18.9 c/генерация | ETA 166 мин
[сетка] 500/1000 | 19.1 c/генерация | ETA 159 мин
[сетка] 525/

### Структурное и текстовое соответствие

*Вес: 1 балл.*

**Постановка.** Не следует считать один CLIP score доказательством точной геометрии либо корректности класса цифры. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Из реальных PNG рассчитайте edge F1 с допуском и CLIP text-image cosine.

**Результат.** Проверяемое состояние «Структурное и текстовое соответствие» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Из реальных PNG рассчитайте edge F1 с допуском и CLIP text-image cosine» и его результат доступен следующему этапу.

In [8]:
# ЗАДАНИЕ 5 — структурное и текстовое соответствие по реальным PNG: edge F1 с допуском + CLIP cosine.
# Edge F1: precision — доля edge-пикселей генерации рядом с контрольным контуром, recall — наоборот;
# допуск = дилатация на TOL_PX. CLIP — независимая текстовая метрика (на CPU, экономия VRAM).
TOL_PX = 6   # допуск на 256px (~0.7 пикселя исходного MNIST)
from transformers import CLIPModel, CLIPProcessor

CLIP_ID = os.getenv('CV_CLIP_MODEL', 'openai/clip-vit-base-patch32')
CLIP = CLIPModel.from_pretrained(CLIP_ID).eval()
CLIP_PROC = CLIPProcessor.from_pretrained(CLIP_ID)
assert next(CLIP.parameters()).device.type == 'cpu', 'CLIP остаётся на CPU (каждый гигабайт VRAM нужен SD)'

@torch.no_grad()
def clip_embed_image(im):
    if not isinstance(im, Image.Image):
        with Image.open(im) as _im:
            im = _im.convert('RGB')
    pv = CLIP_PROC(images=im, return_tensors='pt')['pixel_values']
    e = CLIP.get_image_features(pv)
    return torch.nn.functional.normalize(e, dim=-1)[0].numpy().astype('float32')

@torch.no_grad()
def clip_embed_text(text):
    enc = CLIP_PROC(text=[text], padding=True, truncation=True, return_tensors='pt')
    e = CLIP.get_text_features(**{k: v for k, v in enc.items() if k in ('input_ids', 'attention_mask')})
    return torch.nn.functional.normalize(e, dim=-1)[0].numpy().astype('float32')

# smoke на реальном MNIST: свой класс цифры ближе к своему описанию, чем к 'cat' (3 класса)
TEXT_CLIP = {c: clip_embed_text(f'a photo of a handwritten digit {CLASS_WORDS[c]}') for c in range(10)}
_cat_t = clip_embed_text('a photo of a cat')
_own, _oth = [], []
for _probe in (7, 0, 1):
    _i = int(CTRL_IDX[np.where(_y_train[CTRL_IDX] == _probe)[0][0]])
    _im = Image.fromarray(MNIST_TRAIN.data[_i].numpy().astype(np.uint8)).convert('RGB')
    _e = clip_embed_image(_im)
    _own.append(float(np.dot(_e, TEXT_CLIP[_probe])))
    _oth.append(float(np.dot(_e, _cat_t)))
assert float(np.mean(_own)) > float(np.mean(_oth)), (_own, _oth)

_KERNEL = np.ones((2 * TOL_PX + 1, 2 * TOL_PX + 1), np.uint8)
def edge_f1(gen_edges, ctrl_edges):
    gb, cb = gen_edges > 0, ctrl_edges > 0
    if gb.sum() == 0 or cb.sum() == 0:
        return 0.0
    P = float((gb & (cv2.dilate(ctrl_edges, _KERNEL) > 0)).sum()) / float(gb.sum())
    R = float((cb & (cv2.dilate(gen_edges, _KERNEL) > 0)).sum()) / float(cb.sum())
    return 2 * P * R / (P + R) if (P + R) > 0 else 0.0

def gen_edges_from_png(path):
    with Image.open(path) as im:
        g = np.array(im.convert('L').resize((CTRL_RES, CTRL_RES), Image.BILINEAR))
    return cv2.Canny(cv2.GaussianBlur(g, (3, 3), 0), CANNY_LOW, CANNY_HIGH)

_ctrl_edges_cache = {r['contour_id']: np.array(Image.open(CTRL_DIR / r['file_name'])) for r in ctrl_manifest}
metric_rows, t0 = [], time.time()
for _n, _r in enumerate(gen_manifest):
    _f1 = edge_f1(gen_edges_from_png(_r['path']), _ctrl_edges_cache[_r['contour_id']])
    _cos = float(np.dot(clip_embed_image(_r['path']), TEXT_CLIP[_r['label']]))
    metric_rows.append({'contour_id': _r['contour_id'], 'label': _r['label'], 'setting': _r['setting'],
                        'seed': _r['seed'], 'path': _r['path'],
                        'edge_f1': round(_f1, 4), 'clip_cos': round(_cos, 4)})
    if (_n + 1) % 200 == 0 or (_n + 1) == len(gen_manifest):
        print(f'[метрики] {_n + 1}/{len(gen_manifest)} | {round(time.time() - t0)} c')

GRID_NAMES = [s['name'] for s in SETTINGS if s['name'] != 'base']   # base = baseline, не кандидат в аугментацию
def _agg(name, key):
    vals = [r[key] for r in metric_rows if r['setting'] == name]
    return round(float(np.mean(vals)), 4) if vals else None
metrics_summary = {name: {'scale': next(s['scale'] for s in SETTINGS if s['name'] == name),
                          'cfg': next(s['cfg'] for s in SETTINGS if s['name'] == name),
                          'mean_edge_f1': _agg(name, 'edge_f1'), 'mean_clip_cos': _agg(name, 'clip_cos'),
                          'n_images': sum(1 for r in metric_rows if r['setting'] == name)}
                   for name in [s['name'] for s in SETTINGS]}
SELECTED = max(GRID_NAMES, key=lambda n: metrics_summary[n]['mean_edge_f1'])
WORST_SETTING = min(GRID_NAMES, key=lambda n: metrics_summary[n]['mean_edge_f1'])
metrics_summary['_selection'] = {'selected_for_augmentation': SELECTED, 'worst_setting': WORST_SETTING,
                                 'criterion': 'максимальный/минимальный средний edge F1 (метрика по train-контурам, test не использовался)'}
save_json('lab14_metrics_summary.json', metrics_summary)
(ARTIFACTS / 'lab14' / 'metrics_rows.jsonl').write_text(
    ''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in metric_rows), encoding='utf-8')
_base_f1 = metrics_summary['base']['mean_edge_f1']
_best_f1 = metrics_summary[SELECTED]['mean_edge_f1']
_warn = '' if _base_f1 < _best_f1 else '  ВНИМАНИЕ: baseline не хуже контролируемых настроек — проверьте загрузку ControlNet!'
print(f"ЗАДАНИЕ 5 OK: edge F1 (допуск {TOL_PX}px) и CLIP cosine по {len(metric_rows)} PNG; "
      f"base edge F1 {_base_f1}; лучшая настройка {SELECTED} {_best_f1}; "
      f"худшая {WORST_SETTING} {metrics_summary[WORST_SETTING]['mean_edge_f1']}; "
      f"CLIP smoke (свой класс {_own[0]:.3f} vs cat {_oth[0]:.3f}); {_warn}"
      f"строки -> lab14/metrics_rows.jsonl, сводка -> lab14_metrics_summary.json")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


[метрики] 200/1000 | 11 c
[метрики] 400/1000 | 22 c
[метрики] 600/1000 | 32 c
[метрики] 800/1000 | 43 c
[метрики] 1000/1000 | 54 c
ЗАДАНИЕ 5 OK: edge F1 (допуск 6px) и CLIP cosine по 1000 PNG; base edge F1 0.2578; лучшая настройка s10_cfg75 0.5967; худшая s05_cfg30 0.4281; CLIP smoke (свой класс 0.279 vs cat 0.198); строки -> lab14/metrics_rows.jsonl, сводка -> lab14_metrics_summary.json


### Разнообразие внутри класса

*Вес: 1 балл.*

**Постановка.** Не следует считать один CLIP score доказательством точной геометрии либо корректности класса цифры. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Измерьте попарные различия изображений внутри одного класса и настройки.

**Результат.** Проверяемое состояние «Разнообразие внутри класса» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Измерьте попарные различия изображений внутри одного класса и настройки» и его результат доступен следующему этапу.

In [9]:
# ЗАДАНИЕ 6 — разнообразие внутри класса: попарные различия изображений одной (класс, настройка).
def _gray28(path):
    with Image.open(path) as im:
        return np.asarray(im.convert('L').resize((28, 28), Image.BILINEAR), dtype=np.float32) / 255.0

def _pairwise(gray28_list):
    if len(gray28_list) < 2:
        return None
    X = np.stack(gray28_list).reshape(len(gray28_list), -1)
    n = len(X)
    d2, c = [], []
    for i in range(n):
        for j in range(i + 1, n):
            d2.append(float(np.linalg.norm(X[i] - X[j])))
            c.append(1.0 - float(np.dot(X[i], X[j]) / (np.linalg.norm(X[i]) * np.linalg.norm(X[j]) + 1e-9)))
    return {'mean_l2': round(float(np.mean(d2)), 4), 'mean_cosine_dist': round(float(np.mean(c)), 4),
            'n_pairs': len(d2)}

_by_group = {}
for _r in gen_manifest:
    _by_group.setdefault((_r['label'], _r['setting']), []).append(_r['path'])
diversity = {}
for (_label, _setting), _paths in sorted(_by_group.items()):
    diversity[f'label{_label}/{_setting}'] = _pairwise([_gray28(p) for p in _paths])
_group_keys = [k for k in diversity if not k.startswith('_')]
_per_setting = {}
for _s in SETTINGS:
    vals = [v['mean_l2'] for k, v in diversity.items() if k in _group_keys and v and k.split('/')[1] == _s['name']]
    _per_setting[_s['name']] = round(float(np.mean(vals)), 4) if vals else None
diversity['_per_setting_mean_l2'] = _per_setting
diversity['_definition'] = 'средняя попарная L2 (28x28 серый [0,1]) и попарная косинусная дистанция внутри (класс, настройка)'
save_json('lab14_diversity.json', diversity)
_ex = diversity[_group_keys[0]] if _group_keys else None
print(f"ЗАДАНИЕ 6 OK: разнообразие по {len(_group_keys)} группам (класс x настройка); "
      f"пример: {_ex}; среднее L2 по настройкам: {_per_setting} -> lab14_diversity.json")

ЗАДАНИЕ 6 OK: разнообразие по 50 группам (класс x настройка); пример: {'mean_l2': 13.2679, 'mean_cosine_dist': 0.3017, 'n_pairs': 190}; среднее L2 по настройкам: {'base': 13.0755, 's05_cfg30': 10.8165, 's05_cfg75': 11.5355, 's10_cfg30': 9.0056, 's10_cfg75': 8.8493} -> lab14_diversity.json


### Аудит худших случаев

*Вес: 1 балл.*

**Постановка.** Не следует считать один CLIP score доказательством точной геометрии либо корректности класса цифры. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Сохраните построчные измерения и пять наихудших примеров, не приписывая им вымышленные оценки.

**Результат.** Проверяемое состояние «Аудит худших случаев» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Сохраните построчные измерения и пять наихудших примеров, не приписывая им вымышленные оценки» и его результат доступен следующему этапу.

In [10]:
# ЗАДАНИЕ 7 — аудит худших случаев: построчные измерения + 5 худших по edge F1 (без выдуманных оценок).
import csv, shutil
csv_path = ARTIFACTS / 'lab14_rows_full.csv'
with open(csv_path, 'w', encoding='utf-8', newline='') as f:
    w = csv.writer(f)
    w.writerow(['contour_id', 'label', 'setting', 'seed', 'path', 'edge_f1', 'clip_cos'])
    for r in metric_rows:
        w.writerow([r['contour_id'], r['label'], r['setting'], r['seed'], r['path'], r['edge_f1'], r['clip_cos']])

_worst = sorted(metric_rows, key=lambda r: (r['edge_f1'], r['clip_cos']))[:5]
WORST_DIR = ARTIFACTS / 'lab14' / 'worst5'
WORST_DIR.mkdir(parents=True, exist_ok=True)
worst_manifest = []
for _rank, _r in enumerate(_worst, 1):
    _dst = WORST_DIR / f'{_rank:02d}_c{_r["contour_id"]:03d}_{_r["setting"]}_edgeF1{_r["edge_f1"]}.png'
    shutil.copyfile(_r['path'], _dst)
    worst_manifest.append({**_r, 'rank': _rank, 'saved_as': str(_dst)})
save_json('lab14_worst5.json', {'criterion': '5 минимальных edge F1 (тай-брейк — минимальный clip_cos); значения фактические',
                                'tol_px': TOL_PX, 'rows_total': len(metric_rows), 'worst': worst_manifest})
print(f"ЗАДАНИЕ 7 OK: построчные измерения {len(metric_rows)} строк -> {csv_path.name}; "
      f"5 худших по edge F1 ({[_r['edge_f1'] for _r in _worst]}) -> artifacts/lab14/worst5/ "
      f"(контур/настройка сохранены в lab14_worst5.json; причины — в разделе «Анализ результатов» отчёта)")

ЗАДАНИЕ 7 OK: построчные измерения 1000 строк -> lab14_rows_full.csv; 5 худших по edge F1 ([0.0, 0.0, 0.0, 0.0, 0.0]) -> artifacts/lab14/worst5/ (контур/настройка сохранены в lab14_worst5.json; причины — в разделе «Анализ результатов» отчёта)


### Архитектура downstream-классификатора

*Вес: 1 балл.*

**Постановка.** Структурно похожее изображение может ухудшать downstream classification; качество проверяется на полностью реальном test. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Определите CNN и фиксированную реальную обучающую подвыборку.

**Результат.** Проверяемое состояние «Архитектура downstream-классификатора» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Определите CNN и фиксированную реальную обучающую подвыборку» и его результат доступен следующему этапу.

In [11]:
# ЗАДАНИЕ 8 — архитектура downstream-классификатора (CNN) и фиксированная real-подвыборка.
import torch.nn as nn
import torch.nn.functional as F

MNIST_MEAN, MNIST_STD = 0.1307, 0.3081
class DigitCNN(nn.Module):
    """Небольшой CNN для 28x28: 2 conv-блока + FC; обучается за сотни шагов на CPU/GPU."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 28 -> 14
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 14 -> 7
            nn.Flatten(), nn.Linear(32 * 7 * 7, 10))
    def forward(self, x):
        return self.net(x)

def mnist_normalize(x01):
    return (x01 - MNIST_MEAN) / MNIST_STD

REAL_X = np.stack([MNIST_TRAIN.data[i].numpy().astype(np.float32) / 255.0 for i in REAL_IDX])[:, None]
REAL_Y = _y_train[REAL_IDX].copy()
assert REAL_X.shape == (N_REAL_PER_CLASS * 10, 1, 28, 28) and REAL_X.min() >= 0.0 and REAL_X.max() <= 1.0
_n_params = sum(p.numel() for p in DigitCNN().parameters())
save_json('lab14_classifier_report.json', {
    'arch': 'Conv(1->16,3x3)+ReLU+MaxPool -> Conv(16->32,3x3)+ReLU+MaxPool -> Flatten -> FC(1568->10)',
    'n_parameters': _n_params, 'n_real': len(REAL_IDX),
    'real_indices': REAL_IDX.tolist(),
    'normalization': {'mean': MNIST_MEAN, 'std': MNIST_STD},
    'fixed_subset': 'REAL_IDX из ЗАДАНИЯ 1, один и тот же для real-only и real+synthetic'})
print(f"ЗАДАНИЕ 8 OK: DigitCNN ({_n_params} параметров); real-подвыборка {REAL_X.shape} зафиксирована "
      f"({N_REAL_PER_CLASS}/класс, не пересекается с контурами); нормализация MNIST "
      f"(mean {MNIST_MEAN}, std {MNIST_STD}) -> lab14_classifier_report.json")

ЗАДАНИЕ 8 OK: DigitCNN (20490 параметров); real-подвыборка (1000, 1, 28, 28) зафиксирована (100/класс, не пересекается с контурами); нормализация MNIST (mean 0.1307, std 0.3081) -> lab14_classifier_report.json


### Равнобюджетное обучение с аугментацией

*Вес: 1 балл.*

**Постановка.** Структурно похожее изображение может ухудшать downstream classification; качество проверяется на полностью реальном test. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Обучите real-only и real+synthetic варианты с одинаковым числом обновлений.

**Результат.** Проверяемое состояние «Равнобюджетное обучение с аугментацией» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Обучите real-only и real+synthetic варианты с одинаковым числом обновлений» и его результат доступен следующему этапу.

In [12]:
# ЗАДАНИЕ 9 — равнобюджетное обучение: real-only и real+synthetic, ОДИНАКОВОЕ число optimizer steps.
# Синтетический пул: настройка с лучшим средним edge F1 (выбор по train-контурам, test не использовался).
N_STEPS = 2 if FAST_DEV_RUN else 500
BATCH = 8 if FAST_DEV_RUN else 64
assert BATCH % 2 == 0

def synth_from_png(path):
    """PNG генерации -> тензор 28x28, нормализованный как MNIST (с контрастным растяжением p99)."""
    with Image.open(path) as im:
        g = np.asarray(im.convert('L').resize((28, 28), Image.BILINEAR), dtype=np.float32) / 255.0
    g = np.clip(g / max(float(np.percentile(g, 99)), 1e-3), 0.0, 1.0)   # детерминированное растяжение контраста
    return g

_syn_rows = [r for r in gen_manifest if r['setting'] == SELECTED]
SYN_X = np.stack([synth_from_png(r['path']) for r in _syn_rows])[:, None]
SYN_Y = np.array([r['label'] for r in _syn_rows], dtype=np.int64)
assert len(SYN_X) == len(_syn_rows) and (set(SYN_Y.tolist()) == set(range(10)) or FAST_DEV_RUN)

# фиксированные потоки батчей: real-поток ОБЩИЙ для всех вариантов (те же real-примеры на тех же шагах)
_rs_real = np.random.RandomState(SEED + 777)
REAL_STREAM = [_rs_real.choice(len(REAL_X), size=BATCH, replace=False) for _ in range(N_STEPS)]
_rs_syn = np.random.RandomState(SEED + 888)
SYN_STREAM = [_rs_syn.choice(len(SYN_X), size=BATCH // 2, replace=False) for _ in range(N_STEPS)]

def train_variant(syn_pool, variant_seed):
    """N_STEPS optimizer steps; без syn_pool — real-only; с syn_pool — половина батча синтетика."""
    torch.manual_seed(variant_seed)
    model = DigitCNN().to(DEV)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    losses = []
    for step in range(N_STEPS):
        _take = BATCH // 2 if syn_pool else BATCH
        ridx = REAL_STREAM[step][: _take]
        xb = torch.tensor(REAL_X[ridx], dtype=torch.float32)
        yb = torch.tensor(REAL_Y[ridx], dtype=torch.long)
        if syn_pool:
            SX, SY = syn_pool
            sidx = SYN_STREAM[step]
            xb = torch.cat([xb, torch.tensor(SX[sidx], dtype=torch.float32)])
            yb = torch.cat([yb, torch.tensor(SY[sidx], dtype=torch.long)])
        xb = mnist_normalize(xb).to(DEV)
        yb = yb.to(DEV)
        loss = F.cross_entropy(model(xb), yb)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        if step == 0 or (step + 1) % 25 == 0 or (step + 1) == N_STEPS:
            losses.append({'step': step + 1, 'loss': round(float(loss.item()), 4)})
    return model, losses

@torch.no_grad()
def evaluate(model, X, Y, batch=512):
    model.eval()
    preds = []
    for i in range(0, len(X), batch):
        xb = mnist_normalize(torch.tensor(X[i:i + batch], dtype=torch.float32)).to(DEV)
        preds.append(model(xb).argmax(1).cpu().numpy())
    model.train()
    return np.concatenate(preds)

model_real, log_real = train_variant(None, SEED + 1)
model_syn, log_syn = train_variant((SYN_X, SYN_Y), SEED + 1)   # тот же real-поток + синтетика
torch.save(model_real.state_dict(), ARTIFACTS / 'lab14' / 'ckpt_real_only.pt')
torch.save(model_syn.state_dict(), ARTIFACTS / 'lab14' / 'ckpt_real_syn.pt')
save_json('lab14_training_log.json', {
    'n_steps': N_STEPS, 'batch': BATCH, 'syn_share': '0.5 (половина батча)' ,
    'selected_setting': SELECTED, 'n_synthetic': len(SYN_X),
    'optimizer': 'Adam lr=1e-3', 'real_stream_seed': SEED + 777, 'syn_stream_seed': SEED + 888,
    'real_only': {'losses': log_real}, 'real_synthetic': {'losses': log_syn}})
print(f"ЗАДАНИЕ 9 OK: два варианта по {N_STEPS} optimizer steps (batch {BATCH}); "
      f"real+synthetic: настройка {SELECTED} ({len(SYN_X)} синтетических изображений, половина батча); "
      f"real-поток батчей идентичен в обоих вариантах; финальные loss: real-only {log_real[-1]['loss']}, "
      f"real+synthetic {log_syn[-1]['loss']}; checkpoints -> artifacts/lab14/ckpt_*.pt")

ЗАДАНИЕ 9 OK: два варианта по 500 optimizer steps (batch 64); real+synthetic: настройка s10_cfg75 (200 синтетических изображений, половина батча); real-поток батчей идентичен в обоих вариантах; финальные loss: real-only 0.003, real+synthetic 0.0099; checkpoints -> artifacts/lab14/ckpt_*.pt


### Сравнение на реальном test

*Вес: 1 балл.*

**Постановка.** Структурно похожее изображение может ухудшать downstream classification; качество проверяется на полностью реальном test. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Сохраните точность по классам и confusion matrices на независимом real test.

**Результат.** Проверяемое состояние «Сравнение на реальном test» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Сохраните точность по классам и confusion matrices на независимом real test» и его результат доступен следующему этапу.

In [13]:
# ЗАДАНИЕ 10 — сравнение на независимом real test: точность по классам и confusion matrices.
TEST_LIMIT = 1000 if FAST_DEV_RUN else 10000
TEST_X = np.stack([MNIST_TEST.data[i].numpy().astype(np.float32) / 255.0 for i in range(TEST_LIMIT)])[:, None]
TEST_Y = np.array(MNIST_TEST.targets)[:TEST_LIMIT].copy()

pred_real = evaluate(model_real, TEST_X, TEST_Y)
pred_syn = evaluate(model_syn, TEST_X, TEST_Y)
def _acc(p):
    return round(float((p == TEST_Y).mean()), 4)
def _per_class(p):
    return {c: round(float((p[TEST_Y == c] == c).mean()), 4) for c in range(10)}
def _confusion(p):
    cm = np.zeros((10, 10), dtype=np.int64)
    np.add.at(cm, (TEST_Y, p), 1)
    return cm
acc_real, acc_syn = _acc(pred_real), _acc(pred_syn)
per_class_real, per_class_syn = _per_class(pred_real), _per_class(pred_syn)
cm_real, cm_syn = _confusion(pred_real), _confusion(pred_syn)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), constrained_layout=True)
for ax, cm, name, a in ((axes[0], cm_real, 'real-only', acc_real), (axes[1], cm_syn, 'real+synthetic', acc_syn)):
    im = ax.imshow(cm, cmap='Blues')
    fig.colorbar(im, ax=ax, fraction=0.046)
    ax.set_title(f'{name}: test accuracy {a:.4f}')
    ax.set_xlabel('предсказанный класс')
    ax.set_ylabel('истинный класс')
    ax.set_xticks(range(10)); ax.set_yticks(range(10))
fig.savefig(ARTIFACTS / 'lab14_confusion.png', dpi=150)
plt.close(fig)

_delta = round(acc_syn - acc_real, 4)
_verdict = ('синтетическая аугментация УЛУЧШАЕТ downstream на real test' if _delta > 0 else
            'синтетическая аугментация НЕ улучшает downstream на real test' if _delta == 0 else
            'синтетическая аугментация УХУДШАЕТ downstream на real test')
save_json('lab14_test_results.json', {'test_size': TEST_LIMIT, 'real_only': {'accuracy': acc_real, 'per_class': per_class_real, 'confusion': cm_real.tolist()},
           'real_synthetic': {'accuracy': acc_syn, 'per_class': per_class_syn, 'confusion': cm_syn.tolist()},
           'delta': _delta, 'verdict': _verdict,
           'note': 'test MNIST полностью реальный и не использовался нигде ранее'})
print(f"ЗАДАНИЕ 10 OK: real test {TEST_LIMIT} изображений; real-only {acc_real}; real+synthetic {acc_syn}; "
      f"дельта {_delta:+.4f} — {_verdict}; confusion -> artifacts/lab14_confusion.png, "
      f"детали по классам -> lab14_test_results.json")

C:\Users\eric1\AppData\Local\Temp\ipykernel_3276\1875214915.py:4: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  TEST_Y = np.array(MNIST_TEST.targets)[:TEST_LIMIT].copy()


ЗАДАНИЕ 10 OK: real test 10000 изображений; real-only 0.9388; real+synthetic 0.9426; дельта +0.0038 — синтетическая аугментация УЛУЧШАЕТ downstream на real test; confusion -> artifacts/lab14_confusion.png, детали по классам -> lab14_test_results.json


### Творческое мини-исследование

*Вес: 1 балл, бонус сверх десяти обязательных заданий.*

**Постановка.** Контролируемая абляция позволяет отделить влияние одного фактора от шума выборки. Гипотеза формулируется до просмотра отложенных результатов.

Выполните следующие действия:

1. Сформулируйте собственную предметную гипотезу о поведении модели и измените один фактор, сохраняя те же реальные входы, seed и вычислительный бюджет.
2. Сравните с основным экспериментом, не используя test для выбора настройки; объясните наблюдаемые расхождения.

**Результат.** Отдельные код, измеренная таблица и интерпретация в отчёте.

**Критерий принятия.** Задание считается принятым, если гипотеза проверена на реальных входах, сохранены фактические измерения и показана контролируемая разница. Бонус не является обязательным для Run All.

In [14]:
# БОНУС — творческое мини-исследование (контролируемая абляция, один фактор).
# ГИПОТЕЗА (сформулирована ДО прогона на test): синтетика, порождённая настройкой с ХУДШИМ
# средним edge F1 (слабое следование контуру), при том же бюджете (те же 500 steps, тот же
# real-поток, та же доля синтетики в батче, те же seed-потоки) ДАЁТ ХУЖИЙ downstream accuracy,
# чем синтетика от лучшей настройки. Меняется ровно один фактор: источник синтетики.
if FAST_DEV_RUN:
    print('БОНУС: в FAST-режиме пропущен (полный прогон выполнит абляцию источника синтетики)')
else:
    _worst_rows = [r for r in gen_manifest if r['setting'] == WORST_SETTING]
    WX = np.stack([synth_from_png(r['path']) for r in _worst_rows])[:, None]
    WY = np.array([r['label'] for r in _worst_rows], dtype=np.int64)
    model_worst, log_worst = train_variant((WX, WY), SEED + 1)   # тот же real-поток и seed, другой пул
    torch.save(model_worst.state_dict(), ARTIFACTS / 'lab14' / 'ckpt_real_syn_worst.pt')
    pred_worst = evaluate(model_worst, TEST_X, TEST_Y)
    acc_worst = _acc(pred_worst)
    acc_best = acc_syn
    confirmed = bool(acc_worst < acc_best)
    bonus = {'hypothesis': 'синтетика от настройки с худшим edge F1 даёт меньший downstream accuracy, чем от лучшей (один фактор: источник синтетики)',
             'budget': {'n_steps': N_STEPS, 'batch': BATCH, 'real_stream': SEED + 777, 'syn_stream': SEED + 888},
             'table': {'real_only@500': acc_real,
                       f'real+syn[{SELECTED}]@500': acc_best,
                       f'real+syn[{WORST_SETTING}]@500': acc_worst},
             'confirmed': confirmed,
             'interpretation': (f'accuracy: best {acc_best} vs worst {acc_worst} '
                                f'({"ниже" if acc_worst < acc_best else "не ниже"}); '
                                f'гипотеза {"ПОДТВЕРЖДЕНА" if confirmed else "НЕ подтверждена"}; '
                                'test использован один раз для каждой модели, выбор настройки — по edge F1 train-контуров')}
    save_json('bonus.json', bonus)
    (ARTIFACTS / 'bonus.md').write_text(
        '# Бонус: абляция источника синтетики (edge F1 лучшей vs худшей настройки)\n\n'
        '**Гипотеза (до прогона):** синтетика от настройки с худшим структурным соответствием (edge F1) '
        'даёт меньший downstream accuracy на реальном test при равном бюджете.\n\n'
        '| вариант | accuracy на real test |\n|---|---|\n' +
        ''.join(f'| {k} | {v} |\n' for k, v in bonus['table'].items()) +
        f'\n**Итог:** {bonus["interpretation"]}\n', encoding='utf-8')
    print(f"БОНУС OK: real-only {acc_real}; real+syn[{SELECTED}] {acc_best}; "
          f"real+syn[{WORST_SETTING}] {acc_worst}; {bonus['interpretation']}; -> artifacts/bonus.json + bonus.md")

БОНУС OK: real-only 0.9388; real+syn[s10_cfg75] 0.9426; real+syn[s05_cfg30] 0.9433; accuracy: best 0.9426 vs worst 0.9433 (не ниже); гипотеза НЕ подтверждена; test использован один раз для каждой модели, выбор настройки — по edge F1 train-контуров; -> artifacts/bonus.json + bonus.md


## Анализ результатов

Сопоставьте конфигурации при одинаковых данных, seeds и бюджете. Приведите таблицу фактических метрик и времени, не менее пяти неудачных примеров с объяснением причин, ограничения и проверяемые выводы. Не подменяйте измерения утверждением об успешном запуске.

## Требования к сдаче

Перед сдачей перезапустите ядро и выполните все ячейки по порядку в полном режиме. Приложите конфигурацию, версии библиотек, источники/лицензии данных и модели, checkpoints, таблицу измерений, изображения и краткий анализ ошибок. Для длительного обучения перенесите каталог `artifacts/` на Google Drive и возобновляйте обучение с последнего checkpoint; отсутствие GPU или данных нельзя заменять синтетическими метриками.

## Итоги выполнения

> Заполняется по факту полного прогона (`FAST_DEV_RUN=False`); FAST-прогон в отчёт не включается.

**Протокол.** GPU: NVIDIA GeForce GTX 1050 Ti, 4 ГБ VRAM (cc 6.1); версии: torch 2.14.1+cu126, diffusers 0.39.0, transformers 4.57.6, opencv 5.0.0. Режим: 256 px, 20 шагов UniPC; сетка: base (scale 0) + scale {0.5, 1.0} x CFG {3.0, 7.5}, 5 генераций на контур; downstream: 500 optimizer steps на вариант, batch 64 (real 32 + synthetic 32), DigitCNN 20490 параметров, seed 42; fp16.

**Данные.** MNIST train/test; контуры 200 (20/класс) и real-подвыборка 1000 (100/класс) из train не пересекаются; test 10000 изображений использован только в финальном сравнении; лицензии: MNIST — public domain, SD1.5/ControlNet — CreativeML Open RAIL-M.

**Сетка генераций (одинаковые контуры/семена).**

| настройка | scale | CFG | edge F1 | CLIP cos | разнообразие (L2) |
|---|---|---|---|---|---|
| base | 0.0 | 7.5 | 0.2578 | 0.269 | 13.0755 |
| s05_cfg30 | 0.5 | 3.0 | 0.4281 | 0.2595 | 10.8165 |
| s05_cfg75 | 0.5 | 7.5 | 0.4829 | 0.2768 | 11.5355 |
| s10_cfg30 | 1.0 | 3.0 | 0.4788 | 0.252 | 9.0056 |
| s10_cfg75 | 1.0 | 7.5 | 0.5967 | 0.266 | 8.8493 |

**Источник синтетики (по edge F1, test не использовался):** s10_cfg75 (0.5967); худшая настройка: s05_cfg30 (0.4281).

**Downstream на реальном test (N = 10000).** real-only 0.9388; real+synthetic 0.9426; дельта +0.0038 — синтетическая аугментация улучшает downstream на real test (по классам и confusion — `lab14_test_results.json`, `lab14_confusion.png`).

**Бонус (абляция источника синтетики).** худшая настройка s05_cfg30: accuracy 0.9433 против 0.9426 у лучшей; гипотеза НЕ подтверждена (разница 0.0007 — около 7 изображений из 10000, в пределах шума; структурный edge F1 не предсказывает downstream accuracy) (артефакт: `bonus.json`, `bonus.md`).

**Худшие случаи.** 5 минимальных edge F1: 0.0, 0.0, 0.0, 0.0, 0.0 (файлы `lab14/worst5/`, построчные измерения `lab14_rows_full.csv`).

**Время этапов.** контуры 0.2 c; генерации 20795 c (≈5 ч 47 мин); метрики 54 c; обучение не хронометрировалось (2 варианта × 500 шагов DigitCNN); всего ≈20849 c (≈5.8 ч) без учёта обучения.